# MIG Cement Demand Forecasting & Inventory Optimisation
**A business-first, end-to-end analytical case study**  
**Author:** Oluwadamilare Akinwumi Tijani  
**Source:** MIG's *Demand Forecasting for Cement Across Multiple Sites* case study  
**Working version:** 18 November 2026

I begin with MIG's actual planning decisions and operational constraints. I then investigate the data, develop a mathematically consistent decision framework, test demand forecasting methods and examine what inventory policies can and cannot establish.

**Research status (18 November 2026):** I have executed the analytical notebook against the authorised SQLite source in read-only mode and independently checked the replenishment-policy sensitivity. This file retains the original data outside GitHub and stores no execution outputs. These are historical and conditional experiments, not independently prospective business results.

**Privacy:** I open authorised source data in read-only mode. I do not include raw site-level records, a database copy, site-specific dashboard screenshots, or credentials in this deliverable. I review all outputs before publishing.

## Phase 1 — Define the business problem before any model

MIG operates across multiple construction sites. Cement availability must match planned pours, but daily consumption varies with project schedules, weather and material supply. Manual estimates and rolling **four-week schedules** make it difficult to plan eight weeks ahead. The business challenge is not simply to predict a daily number: it is to reduce the risk of delayed pours **without** creating excess stock, capacity conflicts or unnecessary emergency deliveries.

I keep the case-study objectives unchanged:

| Decision | Original objective | How I evaluate it |
|---|---|---|
| Forecast future consumption | **MAPE ≤ 15%** over a **56-day rolling horizon**, at site level | MAPE on strictly positive observed consumption; also show coverage, MAE, RMSE, WAPE and signed bias on all observations |
| Avoid stockouts when pours are scheduled | **≥ 98% pour readiness** | Fraction of planned pour days with stock available before the pour; label simulated readiness as a *proxy*, not a recorded completion rate |
| Improve stock efficiency | **≥ 20% improvement in stock utilisation** | Compare with a named baseline, report levels and relative change; agree whether higher utilisation or lower average stock is the intended business outcome |
| Reduce waste | **≥ 30% reduction in write-offs** | Require expiry/disposal tonnage and costs; the supplied inventory balances alone cannot prove write-offs |
| Improve decision visibility | **Plotly Dash app** | Local planner/site-manager views of forecast, projected stock, utilisation, reorder alert and scenario assumptions |

**Decision question:** What and how much should MIG order, at each site, and when, while protecting pours and respecting storage constraints?

I will not change the original targets simply because a model fails to reach them.

### 1.1 My operational workflow and decision timing

At an order decision date, I need to distinguish **known information** (past consumption, today's physically verified inventory, outstanding orders, approved current pour-plan snapshots, delivery lead times and approved storage limits) from **unavailable information** (future realised weather, unrecorded plan revisions, future actual deliveries and future consumption).

The case study says MIG currently plans on rolling **four-week** schedules but seeks **56-day** forecasts. A four-week plan cannot automatically be treated as a confirmed eight-week schedule. If no 56-day as-of snapshots exist, my extended-horizon planned-pour experiments are hypothetical information scenarios, not deployable prospective forecasts.

I organise the work in this order:

1. Audit source integrity, units, storage capacities and inventory flow.
2. Investigate planned pours, consumption, schedule interruptions, stock exposure and weather effects.
3. Establish the available information and the reorder/silo mathematics.
4. Only then evaluate time-series and machine-learning forecasts.
5. Compare replenishment policies with explicit lead-time, capacity, baseline and demand assumptions.
6. Build the decision-support app; score **all** original objectives and document any gap in evidence.

## Phase 2 — Audit the actual operational data

I work with the case-study SQLite tables `Operations`, `Sites` and `CementTypes`, not a reconstructed or synthetic replacement. The case-study slides call the core dataset `Cement_Demand`, while my supplied SQLite file stores the daily measurements in `Operations`; I check the real table structure rather than assume a slide is a physical database schema.

The local database should be located next to this notebook as `MIG_Cement_Records.db` (or `MIG_Cement_Records (1).db`). My earlier execution found **32,880 site-day observations, 30 sites and 1,096 calendar days (2022-01-01 to 2024-12-31)**. These are *previously reported values*, which the following cells should reproduce on my machine.

**Install dependencies locally, if needed:** `%pip install pandas numpy matplotlib scikit-learn lightgbm xgboost catboost plotly dash` in a separate Jupyter cell, then restart the kernel as needed. I intentionally avoid executing package installation automatically.

In [ ]:
import sqlite3
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DB_PATH = Path('MIG_Cement_Records.db')
if not DB_PATH.exists():
    DB_PATH = Path('MIG_Cement_Records (1).db')
if not DB_PATH.exists():
    raise FileNotFoundError('Set DB_PATH to your authorised local SQLite database.')
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
plt.rcParams.update({'figure.figsize': (9, 4), 'axes.spines.top': False, 'axes.spines.right': False})

## 3. Database integrity, schema and relational consistency
I first verify SQLite integrity, foreign keys, table sizes and the declared schema. The operational table contains daily measurements; Sites contains metadata and CementTypes the type dictionary. I join site metadata with a many-to-one validation so a join cannot silently duplicate records. I do not display raw sample records.

In [ ]:
# Open the database in read-only mode.
connection = sqlite3.connect(DB_PATH.resolve().as_uri() + '?mode=ro', uri=True)
integrity = connection.execute('PRAGMA integrity_check').fetchone()[0]
foreign_key_errors = len(connection.execute('PRAGMA foreign_key_check').fetchall())

operations = pd.read_sql_query('SELECT * FROM Operations', connection)
sites = pd.read_sql_query('SELECT * FROM Sites', connection)
cement_types = pd.read_sql_query('SELECT * FROM CementTypes', connection)
connection.close()

print('SQLite integrity:', integrity)
print('Foreign key violations:', foreign_key_errors)
print('Operations:', operations.shape)
print('Sites:', sites.shape)
print('Cement types:', cement_types.shape)

In [ ]:
# Check that the tables can be joined safely.
assert integrity == 'ok' and foreign_key_errors == 0
assert sites.site_id.is_unique and cement_types.cement_type.is_unique
assert operations.site_id.isin(sites.site_id).all()
assert operations.cement_type.isin(cement_types.cement_type).all()

sites = sites.rename(columns={'silo_capacity': 'site_capacity'})
cement_data = operations.merge(sites, on='site_id', validate='many_to_one')
assert len(cement_data) == len(operations)
cement_data['date'] = pd.to_datetime(cement_data['date'])
cement_data = cement_data.sort_values(['site_id', 'date']).reset_index(drop=True)

## 4. Structural completeness and physical quantity checks
I check missing values, site-date uniqueness, daily continuity, nonnegative physical quantities and consistency of the two capacity fields. Temperature is allowed to be negative. Zero consumption remains a meaningful observation, not a missing value. I preserve the original records instead of silently dropping unusual values.

In [ ]:
physical_columns = ['consumed_tonnes', 'planned_pour_tonnes',
                    'opening_inventory_tonnes', 'deliveries_tonnes',
                    'closing_inventory_tonnes', 'rain_mm']
site_days = cement_data.groupby('site_id')['date'].nunique()
calendar_days = (cement_data.date.max() - cement_data.date.min()).days + 1
day_gaps = cement_data.groupby('site_id')['date'].diff().dropna()

checks = pd.Series({
    'Missing values': cement_data.isna().sum().sum(),
    'Duplicate site-date rows': cement_data.duplicated(['site_id', 'date']).sum(),
    'Negative physical values': (cement_data[physical_columns] < 0).sum().sum(),
    'Capacity field mismatches': (cement_data.silo_capacity != cement_data.site_capacity).sum(),
    'Nonpositive capacities': (cement_data.site_capacity <= 0).sum(),
    'Sites missing calendar days': (site_days != calendar_days).sum(),
    'Non-daily intervals within sites': (day_gaps != pd.Timedelta(days=1)).sum()
})
print('First date:', cement_data.date.min().date())
print('Last date:', cement_data.date.max().date())
print('Days:', calendar_days, '| Sites:', len(site_days))
print(checks.to_string())
assert (checks == 0).all()

## 5. Inventory Balance and Capacity Validation
I check whether the recorded cement inventory follows the basic inventory balance principle: opening inventory plus deliveries, minus cement consumption, should equal the closing inventory.

Since the quantities are recorded to two decimal places, I allow a tolerance of 0.011 tonnes to accommodate minor rounding differences.

I also perform two additional checks:

- **Inventory continuity:** Verify that the opening inventory for each day matches the previous day's closing inventory at the same site.
- **Storage capacity:** Identify instances where the recorded closing inventory exceeds the site's stated silo capacity.

These checks help me identify inconsistencies and potential operational data quality issues before developing the forecasting models. A record may be mathematically correct but physically unrealistic if the recorded inventory exceeds the available storage capacity.

In [ ]:
# Opening stock + deliveries - consumption = closing stock.
TOLERANCE = 0.011
opening_stock = cement_data.opening_inventory_tonnes
closing_stock = cement_data.closing_inventory_tonnes
available_stock = opening_stock + cement_data.deliveries_tonnes
actual_consumption = cement_data.consumed_tonnes
planned_pours = cement_data.planned_pour_tonnes
balance_error = available_stock - actual_consumption - closing_stock
previous_closing = cement_data.groupby('site_id')['closing_inventory_tonnes'].shift(1)
continuity_error = opening_stock - previous_closing

print('Nonzero balances (2 decimals):', (balance_error.round(2) != 0).sum())
print('Balances outside tolerance:', (balance_error.abs() > TOLERANCE).sum())
print('Maximum absolute error:', round(balance_error.abs().max(), 4))
print('Stock continuity errors:', (continuity_error.abs() > TOLERANCE).sum())

In [ ]:
# Compare recorded closing stock with the site's stated capacity.
capacity_ratio = closing_stock / cement_data.site_capacity
over_capacity = closing_stock > cement_data.site_capacity + TOLERANCE
print('Records above capacity:', over_capacity.sum())
print('Percentage above capacity:', round(over_capacity.mean() * 100, 4))
print('Sites affected:', cement_data.loc[over_capacity, 'site_id'].nunique())
print('Maximum closing/capacity ratio:', round(capacity_ratio.max(), 4))
print(capacity_ratio.quantile([0, .25, .5, .75, .9, .95, .99, 1]))

In [ ]:
# The chart shows aggregate results only.
annual_data = pd.DataFrame({'year': cement_data.date.dt.year,
                            'over_capacity': over_capacity})
annual_rate = annual_data.groupby('year')['over_capacity'].mean() * 100
annual_rate.plot.bar(color='#087f78', rot=0, title='Recorded overcapacity rate by year')
plt.ylabel('Site-day records (%)')
plt.xlabel('Year')
plt.ylim(0, 100)
plt.tight_layout()
plt.show()

### Interpretation and decision
The source is arithmetically consistent within rounding, but a very large capacity ratio makes the reported inventory physically questionable. I cannot infer whether this is a unit mismatch, a deficient capacity definition, off-silo storage or an issue with the case-study data. I will not divide stock by an arbitrary factor, cap the original records, delete affected sites or treat excess stock as waste. For any subsequent capacity-feasible simulation I must document how excess opening stock is handled as a scenario, not a verified correction. I will request reconciliation of units, capacity meaning and delivery records before claiming operational savings.

### 2.1 Operating interpretation: inventory arithmetic is not physical capacity validation

My earlier executed checks found only seven inventory-balance differences rounded to two decimals, each within 0.011 tonnes, but **11,439 records** (34.79%, affecting 15 sites) had closing inventory above reported site capacity; the largest ratio was roughly **172 times capacity**. These are red flags, not proven wasted cement.

I must ask whether the capacity represents one silo, all site storage, another unit of measure, a snapshot that changed over time, or a case-study data construction issue. I **do not** silently scale capacity, truncate observed stock, discard anomalous sites, or classify excess as a write-off. Any capacity-feasible replenishment replay is explicitly a *hypothetical scenario* pending reconciliation.

## 6. Mathematical distinction between planned demand and realised consumption
I check whether recorded cement consumption is nonnegative and does not exceed either planned pours or available stock. Available stock is the opening inventory plus deliveries. If recorded consumption frequently equals the smaller of planned pours and available stock, supply availability may be limiting consumption. This pattern does not prove causation or establish a universal rule for cement demand.

In [ ]:
# Consumption cannot exceed the plan or the available stock in these records.
physical_bound = np.minimum(planned_pours, available_stock)
matches_bound = (actual_consumption - physical_bound).abs() <= TOLERANCE
print('Zero consumption:', (actual_consumption == 0).sum())
print('Positive plan, zero actual:', ((planned_pours > 0) & (actual_consumption == 0)).sum())
print('Zero plan, positive actual:', ((planned_pours == 0) & (actual_consumption > 0)).sum())
print('Actual above plan:', (actual_consumption > planned_pours).sum())
print('Actual above available stock:', (actual_consumption > available_stock + TOLERANCE).sum())
print('Matches physical bound:', matches_bound.sum())
print('Matching percentage:', round(matches_bound.mean() * 100, 4))

## Phase 3 — Diagnose the operational problems before forecasting

### 3.1 Separate planned requirements, material availability and actual consumption

A scheduled pour is a planning requirement; recorded consumption is a realised outcome, potentially affected by weather, delays, stock or site operations. Recorded consumption is not automatically unconstrained cement **demand**.

In the historical records, `consumed_tonnes` never exceeds `planned_pour_tonnes` or recorded opening stock plus deliveries; roughly **84.44%** of rows match `min(planned_pour, available_stock)` within rounding. This is an **empirical pattern** in this dataset, not a general causal equation for cement demand.

I can flag days with an apparently insufficient stock balance but cannot label them confirmed cancellations: I have neither actual pour completion logs nor intraday delivery timestamps. The calculation below is optimistic because it assumes *all same-day deliveries arrive before the pour*.

In [ ]:
# Preliminary operational exposure: diagnostics, not proof of missed pours.
operating = cement_data.copy()
operating['scheduled'] = operating['planned_pour_tonnes'] > 0
operating['recorded_available'] = (operating['opening_inventory_tonnes']
                                   + operating['deliveries_tonnes'])
operating['stock_gap_if_plan_executes'] = np.maximum(
    0, operating['planned_pour_tonnes'] - operating['recorded_available'])
operating['nonexecution_gap'] = np.maximum(
    0, operating['planned_pour_tonnes'] - operating['consumed_tonnes'])
operating['optimistic_stock_ready'] = (operating['recorded_available'] + TOLERANCE
                                       >= operating['planned_pour_tonnes'])
operating['stock_above_recorded_capacity'] = (
    operating['closing_inventory_tonnes'] > operating['site_capacity'] + TOLERANCE)
active = operating.loc[operating['scheduled']]
operational_summary = pd.Series({
    'Scheduled site-days': int(len(active)),
    'Scheduled with zero recorded consumption': int((active.consumed_tonnes == 0).sum()),
    'Potential stock-exposure site-days': int((active.stock_gap_if_plan_executes > TOLERANCE).sum()),
    'Potential stock-exposure tonnes': float(active.stock_gap_if_plan_executes.sum()),
    'Optimistic availability proxy (%)': 100 * float(active.optimistic_stock_ready.mean()),
    'Recorded overcapacity site-days': int(operating.stock_above_recorded_capacity.sum()),
    'Unrealised plan tonnes, all causes': float(operating.nonexecution_gap.sum())
})
print(operational_summary.round(2).to_string())
print('Caution: deliveries are assumed available before pouring; actual timestamps and cancellations are missing.')

### 3.2 Identify which operational signals should inform decisions

I compare the relationship between planned pours and consumption with past-consumption persistence. I then assess the association between heavy rain and non-consumption *among planned pour days*. This guides investigation; it must not be mistaken for a weather-causality experiment.

In [ ]:
# Shift within each site to avoid mixing observations from different sites.
site_consumption = cement_data.groupby('site_id')['consumed_tonnes']
previous_day = site_consumption.shift(1)
previous_week = site_consumption.shift(7)

site_correlations = []
for site_id, consumption in site_consumption:
    site_correlations.append(consumption.autocorr(lag=1))

correlations = pd.Series({
    'Same-day planned pours': actual_consumption.corr(planned_pours),
    'Previous-day consumption': actual_consumption.corr(previous_day),
    'Previous-week consumption': actual_consumption.corr(previous_week),
    'Rainfall (same day)': actual_consumption.corr(cement_data.rain_mm),
    'Mean within-site lag-one autocorrelation': np.mean(site_correlations)
})
print(correlations.to_string())

In [ ]:
correlations.sort_values().plot.barh(color='#397eae', title='Historical associations with recorded consumption')
plt.axvline(0, color='#64748b')
plt.xlim(-1, 1)
plt.xlabel('Correlation coefficient')
plt.tight_layout()
plt.show()

I investigate weather disruption separately on positive-plan days. The 15 mm split is a descriptive exploratory threshold identified in the preliminary analysis, not a calibrated causal threshold. I report the proportion of planned cement consumed and the proportion of zero-use days.

In [ ]:
# Study rain disruption on days when a pour was planned.
active_days = cement_data.loc[planned_pours > 0].copy()
active_days['rain_group'] = np.where(active_days.rain_mm > 15, 'Above 15 mm', 'At most 15 mm')
active_days['realisation_pct'] = 100 * active_days.consumed_tonnes / active_days.planned_pour_tonnes
active_days['zero_consumption_pct'] = (active_days.consumed_tonnes == 0) * 100

weather = active_days.groupby('rain_group').agg(
    records=('consumed_tonnes', 'size'),
    mean_realisation_pct=('realisation_pct', 'mean'),
    zero_consumption_pct=('zero_consumption_pct', 'mean')
)
print(weather.to_string())

In [ ]:
weather[['mean_realisation_pct', 'zero_consumption_pct']].plot.bar(
    color=['#087f78', '#d97706'], rot=0,
    title='Positive-plan days: rainfall and realised consumption')
plt.ylabel('Percent')
plt.xlabel('Observed daily rainfall')
plt.legend(['Mean plan realised', 'Zero-consumption days'])
plt.tight_layout()
plt.show()

### 3.2.1 Cement grade availability is not established

I found that all 30 sites report three different cement grades across their daily history, but the source carries one pooled site-level inventory balance rather than grade-specific opening and closing stocks. I therefore cannot verify whether a site had the *correct grade* for a planned pour, even where its total reported tonnage appears sufficient. For operational procurement, I need grade-level silo inventory, transfers, substitution rules and approved pour-grade requirements.



In [ ]:
grade_by_site = cement_data.groupby('site_id')['cement_type'].nunique()
site_grade_changes = cement_data.groupby('site_id')['cement_type'].transform(
    lambda series: series.ne(series.shift(1))
).sum() - cement_data.site_id.nunique()
print('Sites with all 3 observed grades:', int(grade_by_site.eq(3).sum()))
print('Grade changes between consecutive site-days:', int(site_grade_changes))
print('Separate opening/closing stock per grade: not supplied')
print('Consequence: grade-specific pour readiness cannot be verified.')


### 3.3 Translate observations into actionable questions

I now have four distinct operational questions, each requiring a different intervention:

| Diagnostic | Operational question | Required response or missing evidence |
|---|---|---|
| Scheduled pour, zero consumption | Was the pour cancelled, postponed or blocked by stock? | Actual completion/cancellation logs, weather and job status |
| Recorded stock above declared capacity | Are units or capacity definitions consistent? | Silo inventory by grade, storage allocation, measurement units |
| Limited future plan information | Can an eight-week order actually rely on current plans? | Dated, versioned schedule snapshots and uncertainty beyond week four |
| Stock delivery and consumption timing | Will materials arrive *before* a scheduled pour? | Order timestamps, supplier lead time, delivery receipt times |

A site-level ranking may be explored privately in Jupyter, but I will only publish aggregate outcomes unless the data owner authorises otherwise.

## Phase 4 — Define the mathematics and the forecast-time information boundary

For site \(s\) and day \(t\):

- \(P_{s,t}\): cement planned for scheduled pours.
- \(D_{s,t}\): underlying cement requirement (not directly observed in the dataset).
- \(C_{s,t}\): recorded cement consumed.
- \(I_{s,t}\): physically available stock before demand is served.
- \(Q_{s,t}\): received deliveries.
- \(K_s\): verified site storage capacity.

The historical inventory identity is

\[
 I^{close}_{s,t} = I^{open}_{s,t} + Q_{s,t} - C_{s,t}.
\]

For a simulated policy, fulfilled tonnes must obey \(0\leq F_{s,t}\leq D_{s,t}\) and available stock, and closing stock is opening stock + received deliveries - fulfilled tonnes. Pending orders are tracked separately. The planned-pour readiness proxy is the fraction of scheduled days where stock available **before the pour** covers planned tonnes. This is not the same as a percentile of lead-time demand, nor does it prove that pours completed.

A common *illustrative* reorder calculation is:

\[
 \text{Inventory position} = \text{stock on hand} + \text{outstanding orders},
\]

\[
 \text{order quantity}= \max(0,\min(K_s,\text{planned protection-period requirement})-\text{inventory position}).
\]

I must additionally check capacity **when shipments arrive**, never permit negative stock, and report rejected/unloadable deliveries. Actual ordering costs, supplier constraints, expiry and disposal are not in the source. The lead time of **two days** and buffer of **ten days** come from an earlier exploratory inventory replay; they are *assumptions*, not contractual facts or independently validated optima.

## 8. Forecast-time information contract
I distinguish the information genuinely available before a forecast is issued from information observed afterwards.

| Variable | Strict forecast | Conditional scenario |
|---|---|---|
| Historical consumption, inventory and deliveries | Yes, only through previous day | Yes |
| Calendar and static site capacity/metadata | Yes, assuming stable metadata | Yes |
| Future planned pours | Not verified | Yes, if schedule was available at issue time |
| Future rainfall/temperature | No realised weather | Only dated forecasts or explicit scenarios |
| Future opening stock and deliveries | No realised values | Simulate from initial stock and ordering policy |
| Future closing stock or consumption | Never | Never as predictor |

The brief calls planned pours forward-looking, but the database has no schedule version or issue timestamp. I therefore report a conditional plan scenario separately from the strict historical-information forecast. I do not conflate realised weather or inventory with information available 56 days earlier.

## 9. Mathematical diagnostic: hindsight is not forecasting
I calculate a simple physical upper bound on recorded consumption for the historical 56-day evaluation window (21 October–15 December 2024). Its error measures how closely the observed target follows the physical constraint, not the accuracy of a prospective forecast. I also calculate the prior forecast metric convention: MAE, RMSE, WAPE and mean signed bias use all observations; MAPE uses only positive actuals and reports coverage. Observations with zero consumption must not be deleted from model training or the other error measures.

In [ ]:
# Calculate error metrics consistently for both diagnostic estimates.
def calculate_metrics(actual, predicted):
    error = predicted - actual
    absolute_error = error.abs()
    positive = actual > 0
    mae = absolute_error.mean()
    rmse = np.sqrt((error ** 2).mean())
    wape = 100 * absolute_error.sum() / actual.sum() if actual.sum() > 0 else np.nan
    mape = 100 * (absolute_error[positive] / actual[positive]).mean()
    return pd.Series({
        'Observations': len(actual), 'Positive actuals': positive.sum(),
        'MAE (tonnes)': mae, 'RMSE (tonnes)': rmse,
        'WAPE (%)': wape, 'MAPE positive (%)': mape, 'Bias (tonnes)': error.mean()
    })

In [ ]:
# Use the same final 56 days for both diagnostic estimates.
start_date = pd.Timestamp('2024-10-21')
end_date = start_date + pd.Timedelta(days=55)
test_period = cement_data.date.between(start_date, end_date)
assert test_period.sum() == 56 * cement_data.site_id.nunique()
actual = actual_consumption[test_period]
plan_metrics = calculate_metrics(actual, planned_pours[test_period])
bound_metrics = calculate_metrics(actual, physical_bound[test_period])
comparison = pd.DataFrame({
    'Unadjusted recorded plan': plan_metrics,
    'Hindsight physical bound (not a forecast)': bound_metrics
}).T
print('Diagnostic period:', start_date.date(), 'to', end_date.date())
print(comparison.to_string())

### Interpretation
The physical-bound calculation uses actual future opening stock and deliveries. It can appear to meet the original MAPE target, but it is inadmissible as an eight-week forecasting result. The earlier history-only four-model benchmark and the conditional plan experiments remain distinct. I will not use the same-day closing balance, which algebraically contains the target, or future actual deliveries to claim forecasting success.

### 4.1 Gate to machine learning

The earlier hindsight physical bound produced **6.75% MAPE** on the comparison window, but it used *actual future inventory and deliveries*. I use it to diagnose the recorded target, **never** to claim a valid operational forecast below the original 15% threshold.

The defensible comparison separates:

- **Forecast-time history only:** past observations and known calendar/site information; eligible for a genuine issue-time forecast.
- **Conditional planned-pour forecasts:** assume the full recorded 56-day schedule was known on forecast issue date, an assumption **not established by the source**.
- **Realised-weather diagnostic:** includes future observed rain/temperature, used for retrospective diagnosis **only**.

For actual procurement, my model must be tested using archived as-of schedule/weather inputs and then connected to a verified demand/stock simulator. Until then I will not claim the 15% target is operationally met.

## Phase 5 — Forecasting experiments, after operational diagnosis

I retain the previously prepared 56-day experiments, rather than discard valid work: historical lags, Random Forest, LightGBM, simpler benchmarks and additional conditional models. I separately label history-only, 56-day plan-conditional and realised-weather scenarios. The development windows are **6 May, 1 July and 26 August 2024**; the later comparison starts **21 October 2024**. Since that latter window has already been examined in research, it is **no longer a pristine untouched final test** for future model searches.

I treat MAPE on positive actuals as the original scoring contract. For procurement I also prioritise WAPE, errors on zero days, bias, distributional uncertainty and shortfall risk. A percentage-error optimiser can underestimate demand in ways that make stocking decisions worse.

**The code below is included for reproducibility, not to imply the full 56-day planned schedule really existed at each issue date.**

In [ ]:
site_metadata = sites[['site_id', 'region']]
data = operations.merge(site_metadata, on='site_id', validate='many_to_one')
data['date'] = pd.to_datetime(data['date'])
data = data.sort_values(['site_id', 'date']).reset_index(drop=True)

assert len(data) == len(operations)
assert not data.duplicated(['site_id', 'date']).any()
assert data[['date', 'site_id', 'region', 'consumed_tonnes']].notna().all().all()
assert (data['consumed_tonnes'] >= 0).all()
daily_gaps = data.groupby('site_id')['date'].diff().dropna()
assert (daily_gaps == pd.Timedelta(days=1)).all()
expected_days = (data.date.max() - data.date.min()).days + 1
assert (data.groupby('site_id')['date'].nunique() == expected_days).all()

print('Daily records:', len(data))
print('Sites:', data.site_id.nunique())
print('Coverage:', data.date.min().date(), 'to', data.date.max().date())

In [ ]:
site_consumption = data.groupby('site_id')['consumed_tonnes']

for lag in [1, 7, 14, 28]:
    data[f'consumption_lag_{lag}'] = site_consumption.shift(lag)

for window in [7, 14, 28]:
    data[f'consumption_mean_{window}'] = site_consumption.transform(
        lambda values: values.shift(1).rolling(window).mean()
    )

data['consumption_std_28'] = site_consumption.transform(
    lambda values: values.shift(1).rolling(28).std()
)

history_features = ['consumption_lag_1', 'consumption_lag_7',
                    'consumption_lag_14', 'consumption_lag_28',
                    'consumption_mean_7', 'consumption_mean_14',
                    'consumption_mean_28', 'consumption_std_28']

print('Historical features:', len(history_features))
print('Rows without a full history:', data[history_features].isna().any(axis=1).sum())

In [ ]:
for site_id, site_data in data.groupby('site_id'):
    actual = site_data['consumed_tonnes'].to_numpy()
    lag_one = site_data['consumption_lag_1'].to_numpy()
    assert np.allclose(lag_one[1:], actual[:-1])
    expected = np.convolve(actual, np.ones(28) / 28, mode='valid')
    rolling_mean = site_data['consumption_mean_28'].to_numpy()
    assert np.allclose(rolling_mean[28:], expected[:-1])

origin_data = data.dropna(subset=history_features).copy()
assert len(data) - len(origin_data) == 28 * data.site_id.nunique()
print('Eligible historical origins:', len(origin_data))
print('Lag and rolling-average validation passed.')

In [ ]:
origin_columns = ['site_id', 'region', 'date'] + history_features
weekly_origins = origin_data.loc[origin_data.date.dt.dayofweek == 0, origin_columns].copy()
weekly_origins = weekly_origins.rename(columns={'date': 'forecast_origin'})

last_origin = data.date.max() - pd.Timedelta(days=55)
weekly_origins = weekly_origins[weekly_origins.forecast_origin <= last_origin]

horizons = pd.DataFrame({'horizon': range(1, 57)})
forecast_data = weekly_origins.merge(horizons, how='cross')
forecast_data['target_date'] = forecast_data['forecast_origin'] + pd.to_timedelta(
    forecast_data['horizon'] - 1, unit='D'
)

targets = data[['site_id', 'date', 'consumed_tonnes']].rename(
    columns={'date': 'target_date', 'consumed_tonnes': 'actual_consumption'}
)
forecast_data = forecast_data.merge(targets, on=['site_id', 'target_date'], validate='many_to_one')
print('Forecast origins:', forecast_data.forecast_origin.nunique())
print('Site-origin-horizon examples:', len(forecast_data))

In [ ]:
forecast_data['day_of_week'] = forecast_data.target_date.dt.dayofweek
forecast_data['month'] = forecast_data.target_date.dt.month
forecast_data['is_weekend'] = (forecast_data.day_of_week >= 5).astype(int)

categorical_features = ['site_id', 'region']
calendar_features = ['horizon', 'day_of_week', 'month', 'is_weekend']
feature_columns = categorical_features + history_features + calendar_features
target_column = 'actual_consumption'

print('Total predictors:', len(feature_columns))
print('Predictors:', ', '.join(feature_columns))

In [ ]:
group_columns = ['site_id', 'forecast_origin']
window_sizes = forecast_data.groupby(group_columns)['horizon'].nunique()
assert (window_sizes == 56).all()
assert not forecast_data.duplicated(group_columns + ['horizon']).any()
assert (forecast_data.target_date >= forecast_data.forecast_origin).all()
assert (forecast_data.target_date - forecast_data.forecast_origin).dt.days.eq(
    forecast_data.horizon - 1).all()
assert (forecast_data.groupby(group_columns)[history_features].nunique() == 1).all().all()
assert forecast_data[feature_columns + [target_column]].notna().all().all()
assert target_column not in feature_columns
print('Complete site-origin windows:', len(window_sizes))
print('Missing feature or target values: 0')
print('Forecast structure and information-boundary checks passed.')

In [ ]:
validation_origins = pd.to_datetime(['2024-05-06', '2024-07-01', '2024-08-26'])
comparison_origin = pd.Timestamp('2024-10-21')
evaluation_origins = list(validation_origins) + [comparison_origin]

split_summary = []
for origin in evaluation_origins:
    training = forecast_data[forecast_data.target_date < origin]
    evaluation = forecast_data[forecast_data.forecast_origin == origin]
    assert training.target_date.max() < origin
    assert len(evaluation) == 56 * data.site_id.nunique()

    split_summary.append({
        'Origin': origin.date(),
        'Purpose': 'Historical comparison' if origin == comparison_origin else 'Development',
        'Last training target': training.target_date.max().date(),
        'Evaluation end': evaluation.target_date.max().date(),
        'Training examples': len(training),
        'Evaluation examples': len(evaluation)
    })

print(pd.DataFrame(split_summary).to_string(index=False))

### 2. Add planned cement requirements
I attach the recorded plan for each target day and site. In this conditional experiment, I assume that the entire 56-day schedule is provided at the start of the forecast. I do not add actual future weather, stock or deliveries because these are not known when the forecast is issued. Historical consumption features are frozen at the start of each window.

In [ ]:
plans = data[['site_id', 'date', 'planned_pour_tonnes']].rename(
    columns={'date': 'target_date', 'planned_pour_tonnes': 'planned_pour'})
scenario_data = forecast_data.merge(plans, on=['site_id', 'target_date'], validate='many_to_one')
assert len(scenario_data) == len(forecast_data)
assert scenario_data.planned_pour.notna().all()
assert (scenario_data.planned_pour >= 0).all()
scenario_features = feature_columns + ['planned_pour']
print('Conditional examples:', len(scenario_data))
print('Conditional predictors:', len(scenario_features))
print('Future plan availability: not verified')

### 3. Establish the comparison methods
I retain two history-only references: the historical mean and the previous notebook's fixed LightGBM model. I compare these against the unadjusted plan, a site-specific historical realisation ratio, a simple linear regression model and LightGBM with the plan added. Random Forest is also trained using planned pours. The realisation ratio divides a site's total historical consumption by its total historical planned quantity. It is estimated using only records before the forecast origin. These fixed model settings are chosen before calculating the comparison results.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
import matplotlib.pyplot as plt

model_names = ['Historical mean', 'History-only LightGBM', 'Recorded plan',
               'Historical realisation ratio', 'Plan linear regression',
               'Plan LightGBM', 'Plan Random Forest']
predictions = []

def record_predictions(rows, model_name, predicted):
    result = rows[['site_id', 'forecast_origin', 'target_date', 'horizon',
                   'actual_consumption', 'planned_pour']].copy()
    result['model'] = model_name
    result['prediction'] = np.maximum(0, np.asarray(predicted))
    assert np.isfinite(result.prediction).all()
    predictions.append(result)

### 4. Evaluate simple forecasting methods
I calculate the unadjusted plan, historical mean and historical realisation ratio. For the ratio, I use all completed historical days at the relevant site, not the actual consumption during the evaluation window. Where a site has no positive historical plan, I use the overall historical ratio. I also fit a pooled linear regression using the planned quantity as its only predictor. All these approaches provide useful benchmarks before testing more complex machine learning models.

In [ ]:
for origin in evaluation_origins:
    history = data[data.date < origin]
    evaluation = scenario_data[scenario_data.forecast_origin == origin]
    totals = history.groupby('site_id')[['consumed_tonnes', 'planned_pour_tonnes']].sum()
    overall_ratio = history.consumed_tonnes.sum() / history.planned_pour_tonnes.sum()
    site_ratios = totals.consumed_tonnes / totals.planned_pour_tonnes.replace(0, np.nan)
    ratio = evaluation.site_id.map(site_ratios).fillna(overall_ratio)

    record_predictions(evaluation, 'Historical mean', evaluation['consumption_mean_28'])
    record_predictions(evaluation, 'Recorded plan', evaluation.planned_pour)
    record_predictions(evaluation, 'Historical realisation ratio', evaluation.planned_pour * ratio)

    linear_model = LinearRegression()
    linear_model.fit(history[['planned_pour_tonnes']].to_numpy(), history.consumed_tonnes)
    record_predictions(evaluation, 'Plan linear regression',
                       linear_model.predict(evaluation[['planned_pour']].to_numpy()))

print('Simple benchmarks completed for all four forecast windows.')

### 5. Train the machine learning models
I compare two LightGBM models with identical settings, one with and one without the planned-pour feature. This isolates the effect of adding the recorded schedule. I also test Random Forest using the conditional feature set. Training uses only labels observed before the relevant forecast origin. I encode site and region categories using the training columns and align the evaluation data to those columns.

In [ ]:
for origin in evaluation_origins:
    training = scenario_data[scenario_data.target_date < origin]
    evaluation = scenario_data[scenario_data.forecast_origin == origin]
    assert training.target_date.max() < origin
    y_train = training.actual_consumption

    X_train = pd.get_dummies(training[scenario_features], dtype=float)
    X_test = pd.get_dummies(evaluation[scenario_features], dtype=float)
    X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
    for model_name in ['History-only LightGBM', 'Plan LightGBM']:
        columns = X_train.columns.tolist()
        if model_name == 'History-only LightGBM':
            columns.remove('planned_pour')
        model = LGBMRegressor(n_estimators=200, learning_rate=0.05, num_leaves=31,
                              min_child_samples=30, random_state=42, n_jobs=2, verbosity=-1)
        model.fit(X_train[columns], y_train)
        record_predictions(evaluation, model_name, model.predict(X_test[columns]))

    forest = RandomForestRegressor(n_estimators=40, max_depth=10, min_samples_leaf=20,
                                   max_features=0.7, random_state=42, n_jobs=2)
    forest.fit(X_train, y_train)
    record_predictions(evaluation, 'Plan Random Forest', forest.predict(X_test))
    print('Completed:', origin.date())

### 6. Validate model coverage
I require complete predictions for every site and day across all four windows. I do not remove any zero-consumption days or difficult sites. I keep the original convention that MAPE is calculated only on positive actual consumption, while other measures use every observation.

In [ ]:
results = pd.concat(predictions, ignore_index=True)
assert not results.duplicated(['model', 'site_id', 'forecast_origin', 'target_date']).any()
coverage = results.groupby(['model', 'forecast_origin']).size().unstack(fill_value=0)
assert coverage.shape == (len(model_names), len(evaluation_origins))
assert (coverage == 56 * data.site_id.nunique()).all().all()
print(coverage.to_string())
print('Total predictions:', len(results))

### 7. Compare forecasting accuracy
I calculate MAPE, WAPE, MAE, RMSE and signed bias. I select the best conditional method using the three development windows, then compare the chosen method on the later historical window. This later window has already been inspected during earlier research and is not independent validation. I do not modify model settings to match the 15% target.

In [ ]:
def calculate_metrics(rows):
    actual = rows.actual_consumption
    error = rows.prediction - actual
    positive = actual > 0
    return pd.Series({
        'Observations': len(rows),
        'MAPE coverage (%)': 100 * positive.mean(),
        'MAPE (%)': 100 * (error[positive].abs() / actual[positive]).mean(),
        'MAE (t)': error.abs().mean(),
        'RMSE (t)': np.sqrt((error ** 2).mean()),
        'WAPE (%)': 100 * error.abs().sum() / actual.sum(),
        'Bias (t)': error.mean()
    })

def score_models(rows):
    scores = []
    for model_name, model_rows in rows.groupby('model'):
        score = calculate_metrics(model_rows)
        score.name = model_name
        scores.append(score)
    return pd.DataFrame(scores).sort_values('MAPE (%)')

development = results[results.forecast_origin.isin(validation_origins)]
comparison = results[results.forecast_origin == comparison_origin]
development_scores = score_models(development)
comparison_scores = score_models(comparison)
conditional_models = model_names[2:]
selected_model = development_scores.loc[conditional_models]['MAPE (%)'].idxmin()
print('Development results:')
print(development_scores.round(3).to_string())
print('Historical comparison results:')
print(comparison_scores.round(3).to_string())
print('Selected conditional model:', selected_model)

### 8. Visualise performance against the target
I display both the development and historical comparison MAPE. I distinguish the historical-only references from methods that assume forward-looking plans are available.

In [ ]:
chart_data = pd.DataFrame({'Development': development_scores['MAPE (%)'],
                           'Historical comparison': comparison_scores['MAPE (%)']})
chart_data.plot.bar(figsize=(11, 5), color=['#087f78', '#397eae'], rot=35)
plt.axhline(15, color='#d97706', linestyle='--', label='15% target')
plt.title('Conditional plan scenario versus historical references')
plt.ylabel('MAPE on positive actuals (%)')
plt.legend()
plt.tight_layout()
plt.show()

### 9. Examine the contribution of planned pours
I compare the history-only and plan-informed versions of the same LightGBM model. I also examine the selected conditional method over shorter and longer forecast horizons. This helps me determine whether the additional information is useful across the entire eight-week period.

In [ ]:
history_mape = comparison_scores.loc['History-only LightGBM', 'MAPE (%)']
plan_mape = comparison_scores.loc['Plan LightGBM', 'MAPE (%)']
print('LightGBM MAPE improvement (percentage points):', round(history_mape - plan_mape, 3))
selected = comparison[comparison.model == selected_model].copy()
selected['Horizon band'] = pd.cut(selected.horizon, bins=[0, 7, 14, 28, 56],
                                  labels=['1–7', '8–14', '15–28', '29–56'])
band_scores = []
for band, rows in selected.groupby('Horizon band', observed=True):
    score = calculate_metrics(rows)
    score.name = band
    band_scores.append(score)
print(pd.DataFrame(band_scores).round(3).to_string())

### 10. Assess the original target and operational implications
I retain the original 15% MAPE target and report whether it is reached in this conditional scenario. Even if a conditional model meets the target, it would require actual schedule snapshots and a genuinely unseen prospective evaluation before I could claim operational forecasting success. I also retain the distinction between recorded consumption and underlying demand, since stock shortages may prevent planned pours from taking place. The inventory objectives of 98% readiness, 20% utilisation improvement and 30% write-off reduction require separate evidence and are not established by this forecast experiment.

In [ ]:
selected_mape = comparison_scores.loc[selected_model, 'MAPE (%)']
print('Selected conditional model:', selected_model)
print('Historical comparison MAPE:', round(selected_mape, 3), '%')
print('Conditional scenario target achieved:', selected_mape <= 15)
print('Verified operational forecast target achieved: not established')

### Findings and next steps
I selected Plan Random Forest using development MAPE. Its later comparison MAPE was 22.748%. Adding planned pours to LightGBM improved the historical comparison MAPE by 15.849 percentage points compared with the otherwise identical history-only model.

I interpret this as evidence from a retrospective conditional experiment, not a verified operational forecast. I would request dated plan snapshots and forecast weather inputs, test them prospectively, and investigate the difference between planned demand and stock-constrained consumption before implementing an inventory policy. I will keep the historical benchmark and 15% target unchanged.

### 11. Extend the planned-pour analysis with weather information
I now investigate whether rainfall and temperature provide additional predictive value. I keep this investigation in the same notebook because both planned pours and weather are operational inputs to the same forecasting problem.

I distinguish three experiments using identical LightGBM settings:

| Experiment | Information used | Interpretation |
|---|---|---|
| Plan LightGBM | Historical features, calendar and recorded planned pours | Existing conditional reference |
| Plan + past weather | The same predictors plus weather observed before the forecast origin | Conditional on plan availability; historical weather is known |
| Plan + realised weather | The same plan predictors plus weather actually observed on the target day | Retrospective diagnostic only; not a deployable forecast |

I have no archived weather forecasts or dated schedule snapshots, so I cannot claim that either future input was available at the start of the 56-day period. I do not use future stock or deliveries. I retain the original 15% MAPE objective and evaluate the same four historical windows.

### 12. Prepare historical weather features
I calculate the average rainfall, average temperature and proportion of heavy-rain days over the previous 28 days at each site. I also calculate seven-day average rainfall to capture recent conditions. The 15 mm threshold is retained from my earlier exploratory analysis; it is not a proven causal threshold. I shift all weather observations by one day before calculating the rolling summaries. Every forecast horizon receives the same weather history available at the origin.

In [ ]:
assert data[['rain_mm', 'avg_temp_c']].notna().all().all()
assert (data.rain_mm >= 0).all()
data['heavy_rain'] = (data.rain_mm > 15).astype(int)
site_weather = data.groupby('site_id')

data['rain_mean_7'] = site_weather['rain_mm'].transform(
    lambda values: values.shift(1).rolling(7).mean())
data['rain_mean_28'] = site_weather['rain_mm'].transform(
    lambda values: values.shift(1).rolling(28).mean())
data['temperature_mean_28'] = site_weather['avg_temp_c'].transform(
    lambda values: values.shift(1).rolling(28).mean())
data['heavy_rain_share_28'] = site_weather['heavy_rain'].transform(
    lambda values: values.shift(1).rolling(28).mean())

past_weather_features = ['rain_mean_7', 'rain_mean_28', 'temperature_mean_28',
                         'heavy_rain_share_28']
weather_history = data[['site_id', 'date'] + past_weather_features].rename(
    columns={'date': 'forecast_origin'})
weather_data = scenario_data.merge(weather_history, on=['site_id', 'forecast_origin'],
                                   validate='many_to_one')
assert len(weather_data) == len(scenario_data)
assert weather_data[past_weather_features].notna().all().all()
assert (weather_data.groupby(['site_id', 'forecast_origin'])[past_weather_features]
        .nunique() == 1).all().all()
print('Historical weather features:', len(past_weather_features))
print('Complete examples:', len(weather_data))

### 13. Validate the historical information boundary
I independently check the historical 28-day rainfall feature using the actual preceding 28 daily observations at each site. I also attach target-day weather for a separate diagnostic experiment. It is deliberately excluded from the past-weather model, as observed future rain is not a genuine weather forecast.

In [ ]:
for site_id, site_data in data.groupby('site_id'):
    rain = site_data.rain_mm.to_numpy()
    expected = np.convolve(rain, np.ones(28) / 28, mode='valid')
    observed = site_data.rain_mean_28.to_numpy()[28:]
    assert np.allclose(observed, expected[:-1])

target_weather = data[['site_id', 'date', 'rain_mm', 'avg_temp_c', 'heavy_rain']].rename(
    columns={'date': 'target_date', 'rain_mm': 'realised_rain',
             'avg_temp_c': 'realised_temperature', 'heavy_rain': 'realised_heavy_rain'})
weather_data = weather_data.merge(target_weather, on=['site_id', 'target_date'],
                                  validate='many_to_one')
realised_features = ['realised_rain', 'realised_temperature', 'realised_heavy_rain']
assert len(weather_data) == len(scenario_data)
assert weather_data[realised_features].notna().all().all()
print('Weather feature validation passed.')
print('Realised target-day weather is restricted to the retrospective diagnostic.')

### 14. Compare matched LightGBM models
I use the same model settings and chronological evaluation windows from the planned-pour experiment. For each forecast origin, I train on labels strictly before that date. I retain the plan-only reference to isolate the incremental value of weather information. I compare the past-weather model against the plan-only model on development MAPE; the realised-weather model is excluded from operational model selection.

In [ ]:
weather_feature_sets = {
    'Plan only': scenario_features,
    'Plan + past weather': scenario_features + past_weather_features,
    'Plan + realised weather (diagnostic)': scenario_features + realised_features
}
weather_predictions = []

for origin in evaluation_origins:
    training = weather_data[weather_data.target_date < origin]
    evaluation = weather_data[weather_data.forecast_origin == origin]
    assert training.target_date.max() < origin
    assert len(evaluation) == 56 * data.site_id.nunique()

    for name, features in weather_feature_sets.items():
        X_train = pd.get_dummies(training[features], dtype=float)
        X_test = pd.get_dummies(evaluation[features], dtype=float)
        X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
        model = LGBMRegressor(n_estimators=200, learning_rate=0.05, num_leaves=31,
                              min_child_samples=30, random_state=42, n_jobs=2, verbosity=-1)
        model.fit(X_train, training.actual_consumption)
        predicted = np.maximum(0, model.predict(X_test))
        assert np.isfinite(predicted).all()

        result = evaluation[['site_id', 'forecast_origin', 'target_date', 'horizon',
                             'actual_consumption']].copy()
        result['model'] = name
        result['prediction'] = predicted
        weather_predictions.append(result)
    print('Completed:', origin.date())

### 15. Validate and evaluate weather-informed forecasting
I confirm that all three experiments predict exactly the same site-days. I also compare the reproduced plan-only results with the previous notebook section. The original positive-actual MAPE convention and all-observation error metrics remain unchanged.

In [ ]:
weather_results = pd.concat(weather_predictions, ignore_index=True)
coverage = weather_results.groupby(['model', 'forecast_origin']).size().unstack()
assert coverage.shape == (3, 4)
assert (coverage == 56 * data.site_id.nunique()).all().all()
assert not weather_results.duplicated(['model', 'site_id', 'forecast_origin', 'target_date']).any()

weather_development = weather_results[weather_results.forecast_origin.isin(validation_origins)]
weather_comparison = weather_results[weather_results.forecast_origin == comparison_origin]
weather_development_scores = score_models(weather_development)
weather_comparison_scores = score_models(weather_comparison)
print('Development results:')
print(weather_development_scores.round(3).to_string())
print('Historical comparison results:')
print(weather_comparison_scores.round(3).to_string())
print('Complete predictions:', len(weather_results))

### 16. Compare the value of weather information
I select between the plan-only and past-weather LightGBM models using development MAPE. I report realised-weather performance separately, because it uses information unavailable at the forecast origin. The realised-weather result is a diagnostic estimate of predictive value, not a guaranteed performance ceiling or deployable forecasting result.

In [ ]:
eligible_models = ['Plan only', 'Plan + past weather']
selected_weather_model = weather_development_scores.loc[eligible_models, 'MAPE (%)'].idxmin()
plan_reference_mape = weather_comparison_scores.loc['Plan only', 'MAPE (%)']
past_weather_mape = weather_comparison_scores.loc['Plan + past weather', 'MAPE (%)']
realised_weather_mape = weather_comparison_scores.loc['Plan + realised weather (diagnostic)', 'MAPE (%)']
print('Selected LightGBM specification:', selected_weather_model)
print('Plan-only MAPE:', round(plan_reference_mape, 3))
print('Past-weather MAPE:', round(past_weather_mape, 3))
print('Realised-weather diagnostic MAPE:', round(realised_weather_mape, 3))
print('Past-weather improvement (percentage points):', round(plan_reference_mape - past_weather_mape, 3))
print('Realised-weather improvement (percentage points):', round(plan_reference_mape - realised_weather_mape, 3))
print('Verified operational target achieved: not established')

In [ ]:
chart_data = pd.DataFrame({'Development': weather_development_scores['MAPE (%)'],
                           'Historical comparison': weather_comparison_scores['MAPE (%)']})
chart_data.plot.bar(figsize=(10, 5), color=['#087f78', '#397eae'], rot=15)
plt.axhline(15, color='#d97706', linestyle='--', label='15% target')
plt.ylabel('MAPE on positive actuals (%)')
plt.title('Weather information: conditional forecast versus diagnostic')
plt.legend()
plt.tight_layout()
plt.show()

### 17. Findings and next steps
The matched plan-only LightGBM reproduced its previous historical comparison MAPE of 22.660%. Adding past weather produced 22.767%, while the realised-weather diagnostic produced 19.040%. Using development MAPE, I selected **Plan + past weather** among the two plan-conditional LightGBM specifications. The realised-weather diagnostic is not eligible for operational selection.

I retain the earlier Plan Random Forest benchmark selected on development MAPE. This additional experiment evaluates weather information within LightGBM and does not establish a new overall model winner across all algorithms. I do not claim that the original forecasting objective is operationally achieved without verified plan and weather forecast vintages. I will now investigate inventory policies using clearly stated demand, delivery-time and capacity assumptions. Actual write-off reduction cannot be verified without disposal or expiry data.

### 18. Investigate additional models and zero-consumption days
I investigate XGBoost, CatBoost and a two-stage LightGBM model before concluding the forecasting phase. I use the same four evaluation windows, training cutoffs and error measures as the earlier comparisons. I fix model settings before evaluating the results. I compare each method under two information scenarios: recorded planned pours, and recorded planned pours plus realised weather. The latter is a retrospective diagnostic only, not a deployable forecast. The availability of historical plan snapshots is also unverified.

The two-stage model reflects the fact that a site may consume no cement on a planned day. Its first component estimates the probability of positive consumption. Its second component estimates the amount conditional on consumption occurring. Multiplying the two gives the expected recorded consumption, including the possibility of zero. I also investigate a separate percentage-error-oriented model: a positive-amount LightGBM fitted with absolute-error loss weighted by the inverse of positive consumption. That loss directly reflects the positive-actual MAPE criterion; because it is trained on positive outcomes, it may sacrifice accuracy on zero days and is not an expected-demand estimate. I evaluate all methods on every site-day using all-observation measures as well as MAPE.

In [ ]:
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMClassifier, LGBMRegressor
from sklearn.metrics import brier_score_loss

additional_predictions = []
probability_checks = []
experiments = {
    'Plan conditional': scenario_features,
    'Realised-weather diagnostic': scenario_features + realised_features
}
positive_days = (data.consumed_tonnes > 0).sum()
print('Historical zero-consumption days:', (data.consumed_tonnes == 0).sum())
print('Historical positive-consumption days:', positive_days)

### 19. Train and evaluate the additional models
I use fixed moderate tree sizes and deterministic random seeds for a reproducible comparison. XGBoost uses histogram-based gradient boosting and CatBoost uses native categorical features. For the two-stage model, I fit a classifier on all training outcomes and a quantity regressor only on positive training outcomes. I calculate a probability calibration error using the Brier score, where lower is better. The percentage-error-oriented variant uses the same positive training subset but a different loss. I set its prediction to zero on zero-plan days only when the historical training records show no positive consumption on such days. I never use evaluation outcomes to set predictions or choose thresholds.

In [ ]:
def add_additional(rows, scenario, name, predicted):
    result = rows[['site_id', 'forecast_origin', 'target_date', 'horizon',
                   'actual_consumption']].copy()
    result['scenario'] = scenario
    result['model'] = name
    result['prediction'] = np.maximum(0, np.asarray(predicted))
    assert np.isfinite(result.prediction).all()
    additional_predictions.append(result)

for origin in evaluation_origins:
    training = weather_data[weather_data.target_date < origin]
    evaluation = weather_data[weather_data.forecast_origin == origin]
    assert training.target_date.max() < origin
    assert len(evaluation) == 56 * data.site_id.nunique()
    y_train = training.actual_consumption
    positive = y_train > 0

    for scenario, features in experiments.items():
        X_train = pd.get_dummies(training[features], dtype=float)
        X_test = pd.get_dummies(evaluation[features], dtype=float)
        X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

        xgb = XGBRegressor(n_estimators=200, max_depth=5, learning_rate=0.05,
                           tree_method='hist', n_jobs=2, random_state=42)
        xgb.fit(X_train, y_train)
        add_additional(evaluation, scenario, 'XGBoost', xgb.predict(X_test))

        cat = CatBoostRegressor(iterations=150, depth=6, learning_rate=0.05,
                                cat_features=categorical_features, thread_count=2,
                                random_seed=42, verbose=False, allow_writing_files=False)
        cat.fit(training[features], y_train)
        add_additional(evaluation, scenario, 'CatBoost', cat.predict(evaluation[features]))

        classifier = LGBMClassifier(n_estimators=200, learning_rate=0.05,
                                    num_leaves=31, min_child_samples=30,
                                    random_state=42, n_jobs=2, verbosity=-1)
        classifier.fit(X_train, positive.astype(int))
        probability = classifier.predict_proba(X_test)[:, 1]

        quantity = LGBMRegressor(n_estimators=200, learning_rate=0.05,
                                  num_leaves=31, min_child_samples=30,
                                  random_state=42, n_jobs=2, verbosity=-1)
        quantity.fit(X_train.loc[positive], y_train.loc[positive])
        expected_quantity = probability * np.maximum(0, quantity.predict(X_test))
        add_additional(evaluation, scenario, 'Two-stage expected consumption', expected_quantity)
        probability_checks.append({'Scenario': scenario, 'Origin': origin.date(),
                                   'Brier score': brier_score_loss(
                                       evaluation.actual_consumption > 0, probability)})

        percentage_model = LGBMRegressor(objective='regression_l1', n_estimators=200,
                                          learning_rate=0.05, num_leaves=31,
                                          min_child_samples=30, random_state=42,
                                          n_jobs=2, verbosity=-1)
        percentage_model.fit(X_train.loc[positive], y_train.loc[positive],
                             sample_weight=1 / y_train.loc[positive])
        percentage_prediction = np.maximum(0, percentage_model.predict(X_test))
        if not ((training.planned_pour == 0) & positive).any():
            percentage_prediction[evaluation.planned_pour.to_numpy() == 0] = 0
        add_additional(evaluation, scenario, 'Percentage-error-oriented LightGBM', percentage_prediction)

    print('Completed:', origin.date())

### 20. Check complete coverage and compare results
I compare the models using the original positive-actual MAPE convention. I retain all zero-consumption observations for MAE, RMSE, WAPE and bias. I check that each model produces every required prediction under each scenario. I select the best candidate within each scenario using only the earlier development windows, then report the later historical comparison separately. The previously inspected comparison remains exploratory.

In [ ]:
additional_results = pd.concat(additional_predictions, ignore_index=True)
keys = ['scenario', 'model', 'site_id', 'forecast_origin', 'target_date']
assert not additional_results.duplicated(keys).any()
coverage = additional_results.groupby(['scenario', 'model', 'forecast_origin']).size()
assert len(coverage) == 2 * 4 * 4
assert (coverage == 56 * data.site_id.nunique()).all()
print('Complete additional predictions:', len(additional_results))
print(pd.DataFrame(probability_checks).round(4).to_string(index=False))

additional_summary = []
for scenario, scenario_rows in additional_results.groupby('scenario'):
    earlier = scenario_rows[scenario_rows.forecast_origin.isin(validation_origins)]
    later = scenario_rows[scenario_rows.forecast_origin == comparison_origin]
    development_metrics = score_models(earlier)
    comparison_metrics = score_models(later)
    best = development_metrics.index[0]
    print('Scenario:', scenario)
    print('Selected on development MAPE:', best)
    print('Development results:')
    print(development_metrics.round(3).to_string())
    print('Historical comparison results:')
    print(comparison_metrics.round(3).to_string())
    for name in development_metrics.index:
        additional_summary.append({'Scenario': scenario, 'Model': name,
                                   'Development MAPE': development_metrics.loc[name, 'MAPE (%)'],
                                   'Comparison MAPE': comparison_metrics.loc[name, 'MAPE (%)'],
                                   'Comparison WAPE': comparison_metrics.loc[name, 'WAPE (%)'],
                                   'Comparison bias': comparison_metrics.loc[name, 'Bias (t)']})
additional_summary = pd.DataFrame(additional_summary)

### 21. Evaluate the target and compare with previous experiments
I retain the previously established benchmarks in the comparison. The plan-conditional and realised-weather diagnostic results are ranked separately because their information assumptions differ. A model reaching 15% in the realised-weather diagnostic would not demonstrate a valid forward forecast. I report both the target outcome and broader error measures to avoid selecting a model simply because it improves one metric.

In [ ]:
previous_benchmarks = pd.DataFrame([
    {'Scenario': 'Plan conditional', 'Model': 'Plan Random Forest',
     'Development MAPE': 22.935, 'Comparison MAPE': 22.748,
     'Comparison WAPE': 26.360, 'Comparison bias': -0.279},
    {'Scenario': 'Plan conditional', 'Model': 'Plan LightGBM',
     'Development MAPE': 23.086, 'Comparison MAPE': 22.660,
     'Comparison WAPE': 26.337, 'Comparison bias': -0.150},
    {'Scenario': 'Realised-weather diagnostic', 'Model': 'Plan + realised weather LightGBM',
     'Development MAPE': 20.451, 'Comparison MAPE': 19.040,
     'Comparison WAPE': 17.654, 'Comparison bias': -0.037}
])
final_comparison = pd.concat([previous_benchmarks, additional_summary], ignore_index=True)
final_comparison = final_comparison.sort_values(['Scenario', 'Development MAPE'])
final_comparison['Comparison target met'] = final_comparison['Comparison MAPE'] <= 15
print(final_comparison.round(3).to_string(index=False))
print('Verified operational target achieved: not established')

In [ ]:
for scenario, scenario_scores in final_comparison.groupby('Scenario'):
    scenario_scores.set_index('Model')[['Development MAPE', 'Comparison MAPE']].plot.barh(
        figsize=(10, 5), color=['#087f78', '#397eae'])
    plt.axvline(15, color='#d97706', linestyle='--', label='15% target')
    plt.xlabel('MAPE on positive actuals (%)')
    plt.title(scenario)
    plt.legend()
    plt.tight_layout()
    plt.show()

### 22. Findings and decision — actual re-execution

I reran the analytical code against the original authorised SQLite database on 18 November 2026. In the plan-conditional group, percentage-error-oriented LightGBM was selected on development MAPE of **21.390%** and achieved **22.060%** positive-day MAPE on the later historical comparison (WAPE **27.335%**, mean signed bias **−1.114 t**). The two-stage expected-consumption model produced **22.664%** positive-day MAPE, **26.156%** WAPE and **−0.078 t** bias. The realised-weather hindsight percentage-error model reached **17.885%** MAPE, but that input is not a deployable weather forecast.

Some previous saved Markdown numbers differed slightly across environments. I now use these executed results rather than preserving stale figures. The later comparison period has been inspected in earlier research; it is **not** a pristine independent holdout. The 15% accuracy target was not reached and future plan issue dates remain unavailable.



## Phase 6 — Evaluate actual inventory decisions (exploratory policy replay)

Forecasting alone does not solve the case study. I now compare a no-future-plan historical-mean baseline against a two-stage model that uses the 56-day recorded plan *conditionally*. The latter is an inventory candidate because it estimates expected consumption with comparatively small bias; it was **not** the winner on the original MAPE metric. This is a scenario comparison, not proof that MIG can see those plans prospectively.

I run the same two policies across two alternative **demand interpretations**:

1. **Recorded-consumption replay:** serves observed consumption, which may understate stock-constrained demand.
2. **Scheduled-plan stress test:** tries to serve the full planned tonnage, a service-capacity stress scenario rather than a claim that every planned tonne would have been consumed.

I enforce positive stock, two-day delivery lead time, outstanding orders, capacity-limited receipts, and the same initial stock for both policies. I do **not** write off recorded stock above capacity. Excess initial stock is explicitly marked as **segregated in this hypothetical capacity-feasible simulation**; whether that stock could really be held elsewhere is unverified. I also report rejected incoming deliveries rather than silently allowing simulated stock to overflow.

The previous GitHub replay reported **97.43%** versus **97.76%** scheduled-pour readiness for seasonal-naive versus Random Forest, and **59.77%** versus **60.57%** mean capacity utilisation. Those are *earlier exploratory results on recorded consumption*, not targets achieved and not results of the new two-stage stress test below. I use this section to investigate business performance, not to retrofit claims.

In [ ]:
# Assemble one observation per site, day and policy in the later comparison window.
# The baseline uses only historical consumption features. The candidate assumes
# that the recorded future planned pours were available at the forecast origin.
baseline_forecasts = results.loc[
    (results.forecast_origin == comparison_origin) & (results.model == 'Historical mean'),
    ['site_id', 'forecast_origin', 'target_date', 'prediction']
].copy().assign(policy='History-only mean')
candidate_forecasts = additional_results.loc[
    (additional_results.forecast_origin == comparison_origin) &
    (additional_results.scenario == 'Plan conditional') &
    (additional_results.model == 'Two-stage expected consumption'),
    ['site_id', 'forecast_origin', 'target_date', 'prediction']
].copy().assign(policy='Two-stage plan conditional')
policy_predictions = pd.concat([baseline_forecasts, candidate_forecasts], ignore_index=True)
expected = 56 * cement_data.site_id.nunique()
assert len(baseline_forecasts) == expected and len(candidate_forecasts) == expected
assert not policy_predictions.duplicated(['policy', 'site_id', 'target_date']).any()
print('Forecast observations per policy:', expected)

In [ ]:
def replay_inventory(policy_predictions, historical, scenario='recorded', lead_days=2, buffer_days=10):
    """A hypothetical capacity-feasible, end-of-day ordering policy (not a live optimiser)."""
    if scenario not in {'recorded', 'scheduled'}:
        raise ValueError('Unknown demand scenario')
    columns = ['site_id','date','planned_pour_tonnes','consumed_tonnes',
               'opening_inventory_tonnes','site_capacity']
    joined = policy_predictions.merge(
        historical[columns], left_on=['site_id','target_date'],
        right_on=['site_id','date'], validate='many_to_one')
    records = []
    for (policy, site), group in joined.groupby(['policy','site_id'], sort=True):
        group = group.sort_values('target_date').reset_index(drop=True)
        assert len(group) == 56
        assert (group.target_date.diff().dropna() == pd.Timedelta(days=1)).all()
        capacity = float(group.site_capacity.iloc[0])
        if capacity <= 0:
            raise ValueError('Storage capacity must be positive')
        historical_opening = float(group.opening_inventory_tonnes.iloc[0])
        stock = min(historical_opening, capacity)
        segregated_start = max(0., historical_opening - capacity)
        outstanding = {}  # day index -> order tonnes due for delivery
        forecast = group.prediction.clip(lower=0).to_numpy()
        for i, r in enumerate(group.itertuples(index=False)):
            due = outstanding.pop(i, 0.0)
            accepted = min(due, max(0., capacity-stock))
            rejected = max(0., due-accepted)
            stock += accepted  # received before the day's pour: explicit optimistic timing
            available = stock
            planned = float(r.planned_pour_tonnes)
            required = planned if scenario == 'scheduled' else float(r.consumed_tonnes)
            fulfilled = min(required, stock)
            unfulfilled = required-fulfilled
            stock -= fulfilled
            # Orders placed at END of day. They arrive no earlier than lead_days later.
            # Replenishment horizon starts tomorrow; include lead + safety buffer days.
            protected = forecast[i+1:min(len(group),i+lead_days+buffer_days+1)].sum()
            target = min(capacity, float(protected))
            inventory_position = stock + sum(outstanding.values())
            quantity = max(0.,target-inventory_position) if i+lead_days<len(group) else 0.
            if quantity>0:
                outstanding[i+lead_days] = outstanding.get(i+lead_days,0.)+quantity
            records.append({
                'policy':policy, 'site_id':site, 'date':r.target_date,
                'scenario':scenario, 'capacity':capacity, 'available':available,
                'planned':planned, 'required':required, 'fulfilled':fulfilled,
                'unfulfilled':unfulfilled, 'closing':stock, 'received':accepted,
                'rejected_delivery':rejected, 'ordered':quantity,
                'ready_proxy':(available+0.011>=planned),
                'segregated_initial_stock':segregated_start if i==0 else 0.
            })
    replay = pd.DataFrame(records)
    assert (replay[['available','fulfilled','closing','ordered','received','rejected_delivery']] >= -1e-8).all().all()
    assert (replay.available <= replay.capacity+1e-8).all()
    np.testing.assert_allclose(replay.closing, replay.available-replay.fulfilled, atol=1e-8)
    return replay


def summarize_replay(replay):
    summaries=[]
    for (scenario,policy), g in replay.groupby(['scenario','policy']):
        scheduled = g.planned>0
        summaries.append({
            'Demand scenario':scenario, 'Policy':policy,
            'Scheduled days':int(scheduled.sum()),
            'Pour readiness proxy (%)':100*g.loc[scheduled,'ready_proxy'].mean(),
            'Requirement fill (%)':100*g.fulfilled.sum()/g.required.sum() if g.required.sum()>0 else np.nan,
            'Unmet requirement (t)':g.unfulfilled.sum(),
            'Mean inventory / capacity (%)':100*(g.closing/g.capacity).mean(),
            'Mean closing stock (t)':g.closing.mean(),
            'Orders placed':int((g.ordered>0).sum()),
            'Tonnes ordered':g.ordered.sum(),
            'Unloaded delivery (t)':g.rejected_delivery.sum(),
            'Segregated opening stock (t)':g.segregated_initial_stock.sum(),
            'Simulated overcapacity days':int((g.available>g.capacity+0.011).sum())
        })
    return pd.DataFrame(summaries)

### 6.1 Synthetic policy validation

I test the replenishment mechanics with fabricated data. This is a software consistency check, **not** validation of MIG's actual operations or service targets. It runs before the case-study scenario replay.


In [ ]:
# Synthetic regression check: no confidential records are used in this test.
def test_replenishment_mechanics():
    origin = pd.Timestamp('2024-10-21')
    dates = pd.date_range(origin, periods=56, freq='D')
    sample_forecasts = pd.DataFrame({
        'policy': 'synthetic test', 'site_id': 'SYNTHETIC',
        'target_date': dates, 'prediction': 5.0
    })
    sample_history = pd.DataFrame({
        'site_id': 'SYNTHETIC', 'date': dates,
        'planned_pour_tonnes': 5.0, 'consumed_tonnes': 1.0,
        'opening_inventory_tonnes': 10.0, 'site_capacity': 20.0
    })
    for scenario in ('recorded', 'scheduled'):
        tested = replay_inventory(sample_forecasts, sample_history, scenario=scenario)
        assert len(tested) == 56
        assert tested.received.iloc[0] == 0
        assert tested.received.iloc[1] == 0  # Orders have a 2-day assumed lead time.
        assert tested.received.iloc[2] > 0
        assert tested.closing.ge(0).all()
        assert tested.available.le(tested.capacity + 1e-8).all()
        assert np.allclose(tested.available - tested.fulfilled, tested.closing)
    print('Synthetic inventory checks passed: lead time, balance and capacity.')

test_replenishment_mechanics()


In [ ]:
scenario_replays = pd.concat([
    replay_inventory(policy_predictions, cement_data, scenario='recorded'),
    replay_inventory(policy_predictions, cement_data, scenario='scheduled')
], ignore_index=True)
policy_scorecard = summarize_replay(scenario_replays)
print(policy_scorecard.round(2).to_string(index=False))

# A percentage-point change and a relative change are NOT interchangeable.
for demand_scenario, table in policy_scorecard.groupby('Demand scenario'):
    by_policy = table.set_index('Policy')
    baseline = by_policy.loc['History-only mean']
    candidate = by_policy.loc['Two-stage plan conditional']
    for metric in ['Pour readiness proxy (%)','Mean inventory / capacity (%)']:
        percentage_point_change = candidate[metric]-baseline[metric]
        relative_change = (100*percentage_point_change/baseline[metric]
                           if baseline[metric] else np.nan)
        print(f'{demand_scenario} | {metric}: '
              f'{percentage_point_change:+.2f} pp; {relative_change:+.2f}% relative')
print('These outputs are *scenario results*, not independently verified operating KPIs.')

### 6.2 Reorder policy sensitivity and deployment gates

I do not present ten buffer days or two-day lead time as universally optimal. Before deployment I would run sensitivity analyses across supplier lead times, plan uncertainty, ordering frequencies, capacity reconciliation and demand scenarios. The comparator must be a documented existing operating policy where available; a historical-mean model is only an analytical baseline.

**Decisions that remain blocked:**

- **Physical capacity and usable opening stock**: reconcile suspicious units and storage definitions with site teams.
- **Actual 56-day plan availability**: obtain issue-dated schedules; otherwise limit the prospective plan-driven window to what was genuinely known and add uncertainty beyond it.
- **Supplier execution**: record order dates, agreed lead time, minimum order and unloading/receipt times.
- **Pour success**: obtain completion/cancellation records to distinguish service from an availability proxy.
- **Waste and utilisation definitions**: obtain write-off/expiry logs and agree what “20% utilisation improvement” means.

I would not approve procurement rules until these gates are satisfied. Neither more algorithms nor a polished dashboard can repair missing operational evidence.

### 6.2 Decision-policy experiment after the original replay

I test whether ordering against the full scheduled cement requirement performs better than ordering against expected historical consumption. This is a **hypothetical** 56-day stress test, using the August window for buffer selection and the October window for comparison. The original data remain unchanged.

Both policies start at the smaller of the recorded opening balance and stated storage capacity. Excess historical stock is hypothetically segregated, not written off. Cement grades are pooled because grade-specific balances are unavailable. Deliveries are assumed to arrive before pours, no outstanding orders exist at the forecast origin, and purchase orders are placed at the end of each day. Future recorded pour plans are assumed known; **dated plan snapshots do not exist**, so this is not a deployable prospective strategy. Two-day and three-day leads are tested, with unlimited receipts and a 50-tonne/site/day sensitivity cap. That cap is an **illustrative scenario**, not a verified contractual supplier limit. I must not use these results to claim actual completed pours or procurement savings.



In [ ]:
# Operational sensitivity: all original records remain unchanged.
# Future recorded plans are a CONDITIONAL information scenario, not verified plan vintages.
def policy_replay(ops, origin, signal='plan', requirement='plan',
                  lead_days=2, buffer_days=2, receipt_limit=50.0,
                  opening_rule='historical_capped', site_subset=None,
                  return_trace=False):
    origin = pd.Timestamp(origin)
    subset = ops.loc[ops.date.between(origin, origin + pd.Timedelta(days=55))]
    if site_subset is not None:
        subset = subset.loc[subset.site_id.isin(site_subset)]
    records = []
    for site, group in subset.groupby('site_id'):
        group = group.sort_values('date')
        assert len(group) == 56
        cap = float(group.site_capacity.iloc[0])
        plan = group.planned_pour_tonnes.to_numpy(dtype=float)
        cons = group.consumed_tonnes.to_numpy(dtype=float)
        if signal == 'plan':
            forecast = plan.copy()
        elif signal == 'history_mean_28':
            history = ops.loc[(ops.site_id == site) & (ops.date < origin)].sort_values('date')
            forecast = np.full(56, float(history.consumed_tonnes.tail(28).mean()))
        else:
            raise ValueError('Unknown policy signal')
        opening = float(group.opening_inventory_tonnes.iloc[0])
        if opening_rule == 'historical_capped':
            stock = min(cap, opening)
        elif opening_rule == 'half_capacity':
            stock = .5 * cap
        elif opening_rule == 'empty':
            stock = 0.
        else:
            raise ValueError('Unknown opening rule')
        segregated = max(0., opening - cap) if opening_rule == 'historical_capped' else 0.
        outstanding = {}
        for i in range(56):
            arriving = outstanding.pop(i, 0.)
            received = min(arriving, max(0., cap - stock))
            stock += received  # Receipt before the day's pour: optimistic time assumption.
            available = stock
            required = plan[i] if requirement == 'plan' else cons[i]
            served = min(required, stock)
            stock -= served
            if i + lead_days < 56:
                horizon_end = min(56, i + lead_days + buffer_days + 1)
                protection = float(forecast[i+1:horizon_end].sum())
                target = min(cap, protection)
                inventory_position = stock + sum(outstanding.values())
                order = min(receipt_limit, max(0., target - inventory_position))
                if order > 0:
                    outstanding[i + lead_days] = outstanding.get(i + lead_days, 0.) + order
            else:
                order = 0.
            records.append((site, i, required, plan[i], available, served, stock,
                            order, received, cap, segregated if i == 0 else 0.))
    trace = pd.DataFrame(records, columns=['site_id','day','required','plan',
        'available','served','closing','ordered','received','capacity','opening_excess'])
    np.testing.assert_allclose(trace.available - trace.served, trace.closing, atol=1e-8)
    assert trace.available.le(trace.capacity + 1e-8).all()
    assert trace.closing.ge(-1e-8).all()
    assert trace.ordered.le(receipt_limit + 1e-8).all()
    scheduled = trace.plan > 0
    report = {
      'Origin': str(origin.date()), 'Signal': signal, 'Demand scenario': requirement,
      'Lead days': lead_days, 'Buffer days': buffer_days, 'Receipt limit': receipt_limit,
      'Opening rule': opening_rule, 'Scheduled days': int(scheduled.sum()),
      'Readiness proxy (%)': 100 * float(
          (trace.loc[scheduled,'available']+.011 >= trace.loc[scheduled,'plan']).mean()),
      'Requirement fill (%)': 100 * float(trace.served.sum()/trace.required.sum()),
      'Mean closing/capacity (%)': 100 * float((trace.closing/trace.capacity).mean()),
      'Unfilled tonnes': float((trace.required-trace.served).sum()),
      'Opening excess segregated (t)': float(trace.opening_excess.sum()),
      'Orders placed': int((trace.ordered > 0).sum()),
      'Tonnes ordered': float(trace.ordered.sum())
    }
    return (report, trace) if return_trace else report

def select_policy_buffer(signal, lead_days, receipt_limit):
    trials = []
    for buffer in [0,1,2,3,5,7,10,14]:
        trials.append(policy_replay(cement_data, '2024-08-26', signal,
            lead_days=lead_days, buffer_days=buffer, receipt_limit=receipt_limit))
    scores = pd.DataFrame(trials)
    feasible = scores.loc[scores['Readiness proxy (%)'] >= 98]
    if len(feasible):
        chosen = feasible.sort_values(['Mean closing/capacity (%)','Buffer days']).iloc[0]
    else:
        chosen = scores.sort_values(
            ['Readiness proxy (%)','Mean closing/capacity (%)','Buffer days'],
            ascending=[False,True,True]).iloc[0]
    return int(chosen['Buffer days'])

policy_validation = []
for delivery_limit in [float('inf'), 50.0]:
    for lead in [2, 3]:
        for signal in ['history_mean_28', 'plan']:
            buffer = select_policy_buffer(signal, lead, delivery_limit)
            for origin in ['2024-08-26', '2024-10-21']:
                policy_validation.append(policy_replay(cement_data, origin,
                    signal, lead_days=lead, buffer_days=buffer,
                    receipt_limit=delivery_limit))
policy_validation = pd.DataFrame(policy_validation)
print(policy_validation[['Origin','Signal','Lead days','Buffer days',
    'Receipt limit','Readiness proxy (%)','Requirement fill (%)',
    'Mean closing/capacity (%)']].round(3).to_string(index=False))
print('Conditionally supplied future plans and pooled grades; not a procurement guarantee.')


### 6.3 Test robustness to opening inventory and capacity anomalies

Because many historical inventory balances exceed the stated capacity, I repeat the two-day, 50-tonne schedule scenario using historical capped stock, empty stock and an assumed 50%-full opening stock. I also repeat the test on the subset of 15 sites with no recorded over-capacity event. This checks whether the apparent service improvement is robust, not whether the smaller subset has independently verified physical storage. Results are aggregate only.



In [ ]:
overcap_site_ids = set(cement_data.loc[
    cement_data.closing_inventory_tonnes > cement_data.site_capacity + .011, 'site_id'])
less_anomalous_sites = set(cement_data.site_id.unique()) - overcap_site_ids
opening_sensitivity = []
for opening in ['historical_capped','half_capacity','empty']:
    for label, subset in [('All sites', None),
                          ('Sites without recorded overcapacity', less_anomalous_sites)]:
        result = policy_replay(cement_data, '2024-10-21', signal='plan',
            requirement='plan', lead_days=2, buffer_days=2, receipt_limit=50.0,
            opening_rule=opening, site_subset=subset)
        opening_sensitivity.append({'Group':label, **result})
stock_sensitivity = pd.DataFrame(opening_sensitivity)
print(stock_sensitivity[['Group','Opening rule','Readiness proxy (%)',
    'Requirement fill (%)','Mean closing/capacity (%)']].round(3).to_string(index=False))
assert len(less_anomalous_sites) == 15
print('Historical capacity and grade-level stock remain unreconciled.')


## Phase 7 — Decision dashboard within this notebook

The original brief requires a Plotly Dash application. I keep this demonstration in the **same notebook**, rather than introduce another editable notebook. The dashboard uses only local simulation outputs, including individual site information visible **only in my local Jupyter session**. I do not publish or expose the application externally.

I build a simple site selector and separate forecast / projected stock chart, plus network-wide scenario scorecard. For a professional deployment, I would add genuine forecast issue timestamps, authentication, automated source checks and a safe read-only data connection. I will run the dashboard only after the analysis finishes.

In [ ]:
def launch_local_dashboard():
    """A local-only planner and site-manager dashboard of explicit what-if scenarios."""
    import plotly.express as px
    import plotly.graph_objects as go
    from dash import Dash, dcc, html, Input, Output, dash_table

    scenario_data = scenario_replays.copy()
    site_options = sorted(scenario_data.site_id.unique().tolist())
    app = Dash(__name__)
    app.layout = html.Div([
        html.H2('MIG | Cement planning and inventory intelligence'),
        html.P('EXPLORATORY: capacity, plan issue dates, and delivery lead times require validation.'),
        html.Label('Select a site'),
        dcc.Dropdown(id='site', options=[{'label':str(x),'value':x} for x in site_options],
                     value=site_options[0], clearable=False),
        html.Label('Choose the assumed requirement'),
        dcc.RadioItems(id='demand', options=[
            {'label':'Recorded-consumption replay','value':'recorded'},
            {'label':'Full planned-pour stress test','value':'scheduled'}],
            value='recorded', inline=True),
        dcc.Tabs([
            dcc.Tab(label='Forecasts', children=[dcc.Graph(id='forecast-graph')]),
            dcc.Tab(label='Projected silo inventory', children=[dcc.Graph(id='inventory-graph')]),
            dcc.Tab(label='Reorder alerts and network overview', children=[
                html.P('Suggested orders are scenario outputs, not live purchasing instructions.'),
                dcc.Graph(id='network-graph'),
                dash_table.DataTable(id='orders-table', page_size=12,
                    style_table={'overflowX':'auto'}, sort_action='native')])
        ]),
        html.H4('Aggregate scenario scorecard'),
        dash_table.DataTable(data=policy_scorecard.round(2).to_dict('records'),
                             columns=[{'name':col,'id':col} for col in policy_scorecard.columns],
                             page_size=4, style_table={'overflowX':'auto'})
    ], style={'maxWidth':'1200px','margin':'auto','fontFamily':'Arial','padding':'18px'})

    @app.callback(
        Output('forecast-graph','figure'), Output('inventory-graph','figure'),
        Output('network-graph','figure'), Output('orders-table','data'),
        Output('orders-table','columns'),
        Input('site','value'), Input('demand','value'))
    def update_planner(site_id, demand_scenario):
        selected = scenario_data.loc[
            (scenario_data.site_id==site_id) & (scenario_data.scenario==demand_scenario)]
        predicted = policy_predictions.loc[policy_predictions.site_id==site_id]
        forecast_fig = px.line(predicted, x='target_date', y='prediction', color='policy',
                               title='56-day forecast comparison (tonnes)')
        actual_history = cement_data.loc[
            (cement_data.site_id==site_id) &
            (cement_data.date.between(comparison_origin,
                                      comparison_origin+pd.Timedelta(days=55)))]
        forecast_fig.add_trace(go.Scatter(x=actual_history.date,
                                          y=actual_history.consumed_tonnes,
                                          name='Recorded consumption (hindsight)',
                                          mode='lines',line={'dash':'dot','color':'grey'}))
        forecast_fig.update_layout(xaxis_title='Date',yaxis_title='Tonnes')
        inventory_fig = px.line(selected, x='date', y='closing', color='policy',
                                title='Projected closing stock by policy (tonnes)')
        inventory_fig.add_hline(y=float(selected.capacity.iloc[0]),line_dash='dash',
                                annotation_text='Reported capacity (unreconciled)')
        inventory_fig.update_layout(xaxis_title='Date',yaxis_title='Tonnes')

        proposed = scenario_data.loc[
            (scenario_data.scenario==demand_scenario) &
            (scenario_data.policy=='Two-stage plan conditional')].copy()
        by_site = proposed.groupby('site_id',as_index=False).agg(
            order_tonnes=('ordered','sum'),
            unmet_tonnes=('unfulfilled','sum'))
        network_fig = px.bar(by_site, x='site_id', y='order_tonnes',
                             title='Illustrative proposed order tonnage by site')
        network_fig.update_layout(xaxis_title='Site',yaxis_title='Proposed order tonnes')
        proposed_orders = proposed.loc[proposed.ordered>0,
                                      ['site_id','date','ordered','unfulfilled']].copy()
        proposed_orders['date']=proposed_orders['date'].dt.strftime('%Y-%m-%d')
        proposed_orders=proposed_orders.round(2).sort_values(['date','site_id'])
        cols=[{'name':x.replace('_',' ').title(),'id':x} for x in proposed_orders.columns]
        return forecast_fig, inventory_fig, network_fig, proposed_orders.to_dict('records'), cols
    return app

# I manually launch this local dashboard only after running the full notebook:
# preview = launch_local_dashboard()
# preview.run(host='127.0.0.1', port=8050, jupyter_mode='inline', debug=False)


## Phase 8 — Original objectives assessed after re-execution

I ran the original model and simulation cells against the authorised SQLite database and added the capacity-aware schedule-policy sensitivity above. The central limitation is **operational evidence**, not simply model complexity.

| Objective | Reproduced evidence | Verdict |
|---|---|---|
| **MAPE ≤15% for 56-day forecasts** | Earlier strict history-only Random Forest baseline: 38.45%; in this run, plan-conditional percentage-error LightGBM: **22.060%**; actual future-weather hindsight: **17.885%** | **Not met; no as-of plan evidence** |
| **≥98% pour readiness** | The new schedule-informed policy reaches **98.416%** on the later 56-day window with 2-day lead and assumed 50-tonne/day receipt cap, but **97.195%** on August validation and **96.568%** if the later lead time becomes 3 days | **Only one hypothetical scenario crosses target** |
| **≥20% inventory-utilisation improvement** | Under that later scenario, average closing stock/capacity falls from **50.17%** under the historical-mean comparator to **25.51%** with schedule-informed orders; neither stock level nor case-study utilisation definition is reconciled | **Not verified** |
| **≥30% fewer write-offs** | No verified expiry, disposal or write-off events | **Not measurable** |
| **Plotly Dash application** | Local dashboard prototype and notebook preview function exist | **Prototype only** |

**Source-data diagnosis:** 32,880 site-days; 11,439 closing stock balances above stated storage capacity at 15 sites; all sites use three recorded cement grades but no grade-specific inventory; 9,566 scheduled days have recorded stock below the scheduled requirement even if same-day deliveries arrived first. These observations demand capacity, grade, timing and source-data reconciliation.

The new October 98.42% number **does not establish actual service success**. The August comparison and initial-stock sensitivity demonstrate fragility; no current operating policy or actual pour completion records have been supplied. The final comparison period was examined in earlier work and is not a new independently unseen test.

**My next business action:** request grade-level stock and silo capacity reconciliations, issue-dated pour plans, purchase orders and supplier delivery performance, actual completed/cancelled pours, and documented expiry/write-off costs. Until those data exist, I will not issue operational procurement orders or claim savings.

**Source control and privacy:** This one notebook is proposed on the draft GitHub review branch; published main remains unchanged until accepted. The original SQLite data and site-level execution outputs must never be committed.
